# News headline organizer
Portfolio excerpt of class work by Chigozirim Amaefule. Original selected source is preserved; execution outputs and notebook metadata are omitted. See the portfolio for recorded results and limitations.

Question 8: Advanced - News Aggregator with Database Storage
Build a news aggregator that fetches headlines from the NewsAPILinks to an external site. (API key required).

Tasks:

Write a Python program that:
a. Prompts the user to enter a news topic (e.g., "technology", "sports").
b. Retrieves the top 5 headlines related to the topic using the requests package.
c. Stores the headlines, descriptions, and publication dates in an SQLite database.
d. Reads back the data from the database and displays it to the user in a readable format.
e. Ensures that duplicate headlines are not stored in the database.

In [ ]:
"""Fetch NewsAPI headlines and store them in a local SQLite database."""

import os
import sqlite3
from datetime import date
from pathlib import Path

import requests


# Scripts save beside this file; notebooks save in their current working folder.
DATABASE = Path.cwd() / "news.db"
CATEGORIES = {"business", "entertainment", "general", "health", "science",
              "sports", "technology"}


def initialize_database(connection):
    connection.execute("""
        CREATE TABLE IF NOT EXISTS headlines (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            title_key TEXT NOT NULL UNIQUE,
            description TEXT,
            published_at TEXT
        )
    """)
    connection.commit()


def fetch_headlines(topic, api_key):
    params = {"pageSize": 5}
    if topic.lower() in CATEGORIES:
        params.update(category=topic.lower(), country="us")
    else:
        params["q"] = topic
    response = requests.get(
        "https://newsapi.org/v2/top-headlines",
        params=params,
        headers={"X-Api-Key": api_key},
        timeout=20,
    )
    response.raise_for_status()
    data = response.json()
    if data.get("status") != "ok":
        raise ValueError(data.get("message", "NewsAPI returned an error."))
    return data.get("articles", [])[:5]


def save_headlines(connection, articles):
    """Return database IDs, including IDs of already stored headlines."""
    ids = []
    with connection:
        for article in articles:
            title = (article.get("title") or "").strip()
            if not title or title == "[Removed]":
                continue
            key = " ".join(title.casefold().split())
            connection.execute("""
                INSERT INTO headlines
                    (title, title_key, description, published_at)
                VALUES (?, ?, ?, ?)
                ON CONFLICT(title_key) DO NOTHING
            """, (title, key, article.get("description"),
                  article.get("publishedAt")))
            row = connection.execute(
                "SELECT id FROM headlines WHERE title_key = ?", (key,)
            ).fetchone()
            if row[0] not in ids:
                ids.append(row[0])
    return ids


def display_headlines(rows):
    if not rows:
        print("No headlines found.")
        return
    print("\nTop Headlines:\n")
    for number, row in enumerate(rows, 1):
        print(f"{number}. Title: {row['title']}")
        print(f"   Description: {row['description'] or 'No description available.'}")
        print(f"   Published At: {row['published_at'] or 'Unknown'}\n")


def search_headlines(connection, keyword):
    # instr treats %, _ and other characters as literal text.
    key = " ".join(keyword.casefold().split())
    if not key:
        raise ValueError("Enter a nonempty keyword.")
    return connection.execute("""
        SELECT * FROM headlines WHERE instr(title_key, ?) > 0
        ORDER BY published_at DESC, id DESC
    """, (key,)).fetchall()


def delete_headlines(connection, start, end):
    start_date = date.fromisoformat(start)
    end_date = date.fromisoformat(end)
    if start_date > end_date:
        raise ValueError("Start date must be on or before end date.")
    # Both endpoints are inclusive, using the publication date in UTC.
    with connection:
        cursor = connection.execute("""
            DELETE FROM headlines WHERE date(published_at) BETWEEN ? AND ?
        """, (start_date.isoformat(), end_date.isoformat()))
    return cursor.rowcount


def main():
    connection = sqlite3.connect(DATABASE)
    connection.row_factory = sqlite3.Row
    try:
        initialize_database(connection)
        while True:
            print("\n1. Fetch news  2. Search saved headlines  3. Delete by date  4. Exit")
            choice = input("Choose an option: ").strip()
            try:
                if choice == "1":
                    topic = input("Enter news topic: ").strip()
                    if not topic:
                        raise ValueError("Enter a nonempty news topic.")
                    api_key = os.environ.get("NEWS_API_KEY", "").strip()
                    if not api_key:
                        raise ValueError("Set the NEWS_API_KEY environment variable first.")
                    ids = save_headlines(connection, fetch_headlines(topic, api_key))
                    # Read stored values back, preserving the API result order.
                    rows = [connection.execute(
                        "SELECT * FROM headlines WHERE id = ?", (item_id,)
                    ).fetchone() for item_id in ids]
                    display_headlines(rows)
                    if rows:
                        print("Data has been saved to the database (duplicates skipped).")
                elif choice == "2":
                    display_headlines(search_headlines(
                        connection, input("Enter headline keyword: ")
                    ))
                elif choice == "3":
                    start = input("Start date (YYYY-MM-DD, inclusive): ").strip()
                    end = input("End date (YYYY-MM-DD, inclusive): ").strip()
                    count = delete_headlines(connection, start, end)
                    print(f"Deleted {count} headline(s).")
                elif choice == "4":
                    print("Goodbye!")
                    break
                else:
                    print("Choose 1, 2, 3, or 4.")
            except (requests.RequestException, ValueError, sqlite3.Error) as error:
                print(f"Error: {error}")
    finally:
        connection.close()


if __name__ == "__main__":
    try:
        main()
    except (KeyboardInterrupt, EOFError):
        print("\nGoodbye.")
